# Case Study: Cyclistic Bike-Share

**How do annual members and casual riders use Cyclistic bikes differently?**<br>
*Google Data Analytics Professional Certificate – Course 8 Capstone*

**Shaine Meister**<br>
**March 28, 2026**

---


## Introduction
This notebook extends the main Cyclistic case study and documents the end-to-end technical workflow used to build an analysis-ready trip dataset for downstream analysis.

Cyclistic is a bike-share program in Chicago with more than 5,800 bicycles and 600 docking stations. The company offers traditional bikes as well as assistive options such as reclining bikes, hand tricycles, and cargo bikes. Casual riders buy single-ride or full-day passes, while annual members purchase yearly memberships.

Cyclistic's finance team has shown that annual members are more profitable than casual riders. To support strategy work in the main case study, this notebook focuses on the backend preparation steps required to ingest raw trip files, validate data quality, clean invalid records, standardize spatial features, enrich trips with hourly weather context, and export a reusable final dataset.


**High-level workflow covered in this notebook**
1. **Prepare** *(Section 2)*
    - Load monthly Divvy trip CSV files for the configured `start_yyyymm` to `end_yyyymm` range.
    - Validate requested files against available S3 objects, download ZIP archives, and extract monthly CSV files.
    - Combine all monthly files into one consolidated dataset.
    - Standardize core data types and run basic schema/readiness checks.

2. **Process** *(Section 3)*
    - Engineer time-based features and apply rule-based data validation checks.
    - Separate records into `clean_data` and `dirty_data` for traceable cleanup.
    - Convert trip coordinates into stable vector-mapped location keys.
    - Merge hourly weather data into trip records using nearest-station matching and audit the merge results.
    - Export the final enriched analytical dataset for downstream analysis and reporting.

This notebook captures the full preparation and processing pipeline that turns raw Cyclistic trip files into a validated, spatially normalized, weather-enriched analytical dataset.

Note: *The following case-study phases are not performed in this notebook: Ask, Analyze, Share, and Act.*

## 2. Prepare
**Data location**  
Public Cyclistic (Divvy) trip data: [Divvy TripData](https://divvy-tripdata.s3.amazonaws.com/index.html "https://divvy-tripdata.s3.amazonaws.com/index.html")<br>
Weather Data: [Open-Meteo](https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv "https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv")<br>
In this section, the notebook builds the raw trip dataset by validating the requested monthly Divvy files against the public S3 bucket, downloading ZIP archives when needed, extracting the monthly CSV files, and combining them into a single DataFrame.

**What this section does**
1. Validate the configured `start_yyyymm` to `end_yyyymm` month range against available Divvy S3 objects.
2. Download missing ZIP archives to a local cache and extract their CSV contents.
3. Concatenate all monthly CSV files into one consolidated dataset.
4. Standardize `started_at` as datetime and sort records chronologically.
5. Run a quick schema sanity check across extracted monthly files.

**How the data is organized** *(sample)* 
<div style="font-size: 10px;">

| `ride_id` | `rideable_type` | `started_at` | `ended_at` | `start_station_name` | `start_station_id` | `end_station_name` | `end_station_id` |
|---|---|---|---|---|---|---|---|
| Unique trip identifier | Bike type used | Trip start timestamp | Trip end timestamp | Origin station name | Origin station ID | Destination station name | Destination station ID |

</div>

The combined trip dataset is expected to contain approximately 5-6 million rows, depending on the selected date range.

**ROCCC verification**  
- Reliable: Collected by Cyclistic's own system.  
- Original: First-party trip data.  
- Comprehensive: Covers every ride in the system.  
- Current: Uses the configured recent month range.  
- Cited: Licensed for analysis (Motivate International).  

**Licensing, privacy, and security**  
Data is public under the Divvy data license. No personally identifiable information is included, so privacy is preserved. Files are cached locally for reproducible reruns of the notebook.

**Prepare outputs produced here**  
- `df`: consolidated raw trip dataset with `started_at` parsed and sorted.  
- `schema_check`: quick cross-file schema QA summary.  

**Data integrity check**  
A quick preview of row structure and column consistency is displayed below before the notebook moves to the next section.

In [12]:
# Standard library imports
# calendar: monthrange helper; derives the last day of any month for dynamic weather date bounds
# io: in-memory byte/string stream; wraps weather CSV text for pd.read_csv
# os: file-system path utilities (available for downstream cells if needed)
# re: regular expressions; extracts ZIP filenames from the S3 XML bucket listing
# zipfile: ZIP archive reading and member extraction for Divvy monthly trip files
# pathlib.Path: object-oriented path handling for local cache directories
# urllib.request: urlopen streams HTTP for S3 XML and weather CSV; urlretrieve downloads ZIPs to disk

# Third-party libraries
# numpy: array operations for boolean dirty-data masks and coordinate grid math
# pandas: core DataFrame library for trip records, weather tables, and grouped output

import calendar
import io
import os
import re
import zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import numpy as np
import pandas as pd

# Shared path configuration (define once and reuse throughout notebook).
PROJECT_ROOT = Path("/home/stubb/Code/articles/case-study_bike-share-success")
OUTPUT_DIR = PROJECT_ROOT

# Edit these two values to control which months are loaded.
# Format: YYYYMM. Range is inclusive on both ends.
start_yyyymm = "202503"
end_yyyymm = "202504"

# Configure optional Excel export (CSV is always exported).
export_xlsx = False  # Set to True to also write an .xlsx file.

In [13]:
# Prepare phase: data loading overview
# 1) Configure S3 source URLs and local ZIP/CSV cache directories.
# 2) Set target YYYYMM range; build the expected ZIP filename list and verify against S3 availability.
# 3) Download uncached ZIPs and extract monthly CSVs to disk.
# 4) Load all monthly CSVs into df; normalize started_at and sort rows chronologically.
# 5) Fetch Open-Meteo hourly weather CSV once, cache it locally, and parse it into two DataFrames.
# 6) Print output summary.

# 1) S3 source URLs and local cache configuration
# Base URL for the Divvy public S3 bucket — all ZIP filenames are appended directly to this root.
base_url = "https://divvy-tripdata.s3.amazonaws.com/"
# S3 ListObjectsV2 XML endpoint; more reliable than the HTML index for parsing available file keys.
list_url = base_url + "?list-type=2"

# Workspace root for this project; cache subdirectories are created relative to this path.
data_path = PROJECT_ROOT
zip_dir = data_path / "divvy_zip"         # Local cache directory for downloaded Divvy ZIP archives
extract_dir = data_path / "divvy_csv"     # Local cache directory for extracted Divvy monthly CSV files
weather_dir = data_path / "weather_csv"   # Local cache directory for Open-Meteo weather CSV responses
zip_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)
weather_dir.mkdir(parents=True, exist_ok=True)

# 2) YYYYMM range setup and S3 availability verification
# iter_yyyymm: generator that yields every YYYYMM string in a closed [start, end] interval.
# Increments the month counter and rolls December (12) over to January (1) of the next year
# so that multi-year ranges are handled correctly without relying on dateutil or pandas.
def iter_yyyymm(start_yyyymm, end_yyyymm):
    """Yield consecutive YYYYMM strings from start to end (inclusive), handling December-to-January rollover."""
    y, m = int(start_yyyymm[:4]), int(start_yyyymm[4:6])
    end_y, end_m = int(end_yyyymm[:4]), int(end_yyyymm[4:6])

    while (y < end_y) or (y == end_y and m <= end_m):
        yield f"{y:04d}{m:02d}"
        m += 1
        if m > 12:   # December rolls over to January of the following year
            y += 1
            m = 1

# Build the full list of expected ZIP filenames for every month in the requested range.
desired_zip_files = [f"{yyyymm}-divvy-tripdata.zip" for yyyymm in iter_yyyymm(start_yyyymm, end_yyyymm)]

# S3 availability check
# Fetch the bucket's XML object listing and extract all valid YYYYMM ZIP keys using the file pattern.
# Each available file appears as <Key>YYYYMM-divvy-tripdata.zip</Key> in the XML response body.
with urlopen(list_url) as response:
    listing_xml = response.read().decode("utf-8", errors="ignore")

available_zip_files = set(re.findall(r"<Key>(20\d{4}-divvy-tripdata\.zip)</Key>", listing_xml))

# Intersection: only process months that fall within the requested range AND exist in S3.
zip_files = [z for z in desired_zip_files if z in available_zip_files]
# Any month in the requested range but absent from S3 is recorded separately and reported below.
missing_zip_files = [z for z in desired_zip_files if z not in available_zip_files]

if not zip_files:
    raise ValueError(
        f"No Divvy monthly ZIP files found in S3 for range {start_yyyymm} to {end_yyyymm}."
    )

print(f"Requested months: {len(desired_zip_files)}")
print(f"Available ZIP files to process: {len(zip_files)}")
if missing_zip_files:
    print(f"Missing months in S3 (skipped): {len(missing_zip_files)}")
    print("Examples:", missing_zip_files[:5])

# 3) ZIP download and CSV extraction
extracted_csv_paths = []
for zip_name in zip_files:
    zip_path = zip_dir / zip_name
    zip_url = base_url + zip_name

    # Skip download if the ZIP is already cached locally to avoid redundant network requests.
    if not zip_path.exists():
        print(f"Downloading: {zip_name}")
        urlretrieve(zip_url, zip_path)

    # Derive the expected flat CSV filename from the ZIP name (e.g. 202503-divvy-tripdata.csv).
    expected_csv_name = zip_name.replace(".zip", ".csv")
    expected_csv_path = extract_dir / expected_csv_name

    # Skip extraction if the CSV was already unpacked in a previous run.
    if not expected_csv_path.exists():
        with zipfile.ZipFile(zip_path, "r") as zf:
            csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
            if not csv_members:
                raise ValueError(f"No CSV found in ZIP: {zip_name}")

            member = csv_members[0]
            zf.extract(member, path=extract_dir)

            # Some ZIPs embed the CSV inside a subdirectory (e.g. "subdir/YYYYMM-*.csv").
            # Rename the extracted file to a flat path in extract_dir for consistent access.
            extracted_member_path = extract_dir / member
            if extracted_member_path != expected_csv_path:
                extracted_member_path.replace(expected_csv_path)

    extracted_csv_paths.append(expected_csv_path)

# Plain filename list retained for compatibility with downstream notebook references.
csv_files = [p.name for p in extracted_csv_paths]

# 4) Load and concatenate monthly CSVs into a single trip DataFrame
df_list = [pd.read_csv(p) for p in extracted_csv_paths]
df = pd.concat(df_list, ignore_index=True)

# Parse started_at to datetime and sort rows chronologically; required for time-based
# feature engineering and consistent sliding-window operations in the Process section.
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df = df.sort_values("started_at").reset_index(drop=True)

# 5) Weather data import (Open-Meteo archive API)
# Cache the weather CSV locally so repeated notebook runs reuse the prior download,
# matching the same cache-first behavior used for Divvy trip files.
# Date bounds are derived automatically from start_yyyymm / end_yyyymm:
#   weather_start_date = last day of the month immediately before start_yyyymm
#   weather_end_date   = last day of end_yyyymm

start_y, start_m = int(start_yyyymm[:4]), int(start_yyyymm[4:6])
prev_m = start_m - 1
prev_y = start_y
if prev_m == 0:   # January rolls back to December of the prior year
    prev_m = 12
    prev_y -= 1
weather_start_date = f"{prev_y:04d}-{prev_m:02d}-{calendar.monthrange(prev_y, prev_m)[1]:02d}"

end_y, end_m = int(end_yyyymm[:4]), int(end_yyyymm[4:6])
weather_end_date = f"{end_y:04d}-{end_m:02d}-{calendar.monthrange(end_y, end_m)[1]:02d}"

weather_file = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude=41.65,42.10&longitude=-87.85,-87.40"
    f"&start_date={weather_start_date}&end_date={weather_end_date}"
    f"&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,"
    f"wind_speed_10m,wind_direction_10m,cloud_cover"
    f"&timezone=America/Chicago&format=csv"
)
weather_cache_path = weather_dir / f"open_meteo_{weather_start_date}_{weather_end_date}.csv"

if not weather_cache_path.exists():
    print(f"Downloading weather cache: {weather_cache_path.name}")
    urlretrieve(weather_file, weather_cache_path)

# Read the cached weather response and split it into individual lines so the
# dual-table boundary can be detected before passing each slice to pd.read_csv.
lines = weather_cache_path.read_text(encoding="utf-8", errors="ignore").splitlines()

# Locate the index of the second 'location_id' header row (index 0 is always Table 1's header).
# Everything before split_idx belongs to Table 1; from split_idx onward is Table 2.
split_idx = next(
    i for i in range(1, len(lines))
    if lines[i].strip().startswith("location_id")
)

# Rejoin each slice with newlines so pd.read_csv receives a properly row-delimited CSV string.
weather_locations_csv = "\n".join(lines[:split_idx])
weather_observations_csv = "\n".join(lines[split_idx:])

# Parse each table slice into its own DataFrame for independent use in downstream analysis.
weather_locations_df = pd.read_csv(io.StringIO(weather_locations_csv))
weather_df = pd.read_csv(io.StringIO(weather_observations_csv))

# 6) Output summary
print(f"CSV files loaded: {len(extracted_csv_paths)}")
print(f"Rows loaded: {len(df):,}")
display(df.head())

print("\nWeather locations:")
display(weather_locations_df.head())

print("Weather observations:")
display(weather_df.head())


Requested months: 2
Available ZIP files to process: 2
CSV files loaded: 2
Rows loaded: 669,496


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,5FBF1BFE4C72756F,classic_bike,2025-02-28 08:06:36.009,2025-03-01 09:06:25.127,Orleans St & Hubbard St,636,NaN,NaN,41.890028,-87.636618,NaN,NaN,member
1,FDA49BB617F644B4,classic_bike,2025-02-28 12:29:55.028,2025-03-01 13:29:33.874,Sheridan Rd & Montrose Ave,TA1307000107,NaN,NaN,41.961670,-87.654640,NaN,NaN,casual
2,8B20E2B0D8603957,classic_bike,2025-02-28 13:53:38.438,2025-03-01 14:53:31.545,DuSable Lake Shore Dr & Monroe St,13300,NaN,NaN,41.880958,-87.616743,NaN,NaN,casual
3,B7FC6F4C1F021136,classic_bike,2025-02-28 13:58:06.709,2025-03-01 14:58:00.064,Broadway & Granville Ave,15571,NaN,NaN,41.994780,-87.660285,NaN,NaN,member
4,F837FF70199A73B3,classic_bike,2025-02-28 14:02:54.583,2025-03-01 15:02:34.239,Phillips Ave & 83rd St,582,NaN,NaN,41.744531,-87.565060,NaN,NaN,casual



Weather locations:


,location_id,latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation
0,0,41.65202,-87.78903,215.0,-18000,America/Chicago,GMT-5
1,1,42.07381,-87.37610,174.0,-18000,America/Chicago,GMT-5


Weather observations:


,location_id,time,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),rain (mm),snowfall (cm),wind_speed_10m (km/h),wind_direction_10m (°),cloud_cover (%)
0,0,2025-02-28T00:00,1.1,76,0.0,0.0,0.0,9.0,244,89
1,0,2025-02-28T01:00,0.8,77,0.0,0.0,0.0,10.5,235,97
2,0,2025-02-28T02:00,0.6,78,0.0,0.0,0.0,10.0,231,53
3,0,2025-02-28T03:00,0.4,79,0.0,0.0,0.0,12.7,230,0
4,0,2025-02-28T04:00,0.4,79,0.0,0.0,0.0,12.9,221,26


In [14]:
# Optional QA: quick schema sanity check across extracted monthly CSV files.
# Purpose: catch obvious structural drift before concatenation by comparing
# column counts and a small dtype sample from each file.
# Limitations: this does not validate full column names, column order, or
# late-file type issues because it reads only the first 5 rows per file.
schema_rows = []
for csv_path in extracted_csv_paths:
    tmp = pd.read_csv(csv_path, nrows=5)
    schema_rows.append({
        "file": csv_path.name,
        "columns": len(tmp.columns),           # Expected to match across all months
        "sample_dtypes": ", ".join(tmp.dtypes.astype(str).head(5).tolist()),  # Quick sample, not full-schema validation
    })

schema_check = pd.DataFrame(schema_rows).sort_values("file").reset_index(drop=True)
display(schema_check.head(12))

# A single unique column-count value is a useful early signal, not a complete schema guarantee.
print("Unique column counts across files:", sorted(schema_check["columns"].unique()))
print("Quick schema sanity check complete.")


,file,columns,sample_dtypes
0,202503-divvy-tripdata.csv,13,"object, object, object, object, float64"
1,202504-divvy-tripdata.csv,13,"object, object, object, object, object"


Unique column counts across files: [np.int64(13)]
Quick schema sanity check complete.


## 3. Process
**Tools of choice**<br>
Python with pandas and numpy - scalable, reproducible, and suitable for large trip data and downstream enrichment workflows.

This section covers the full processing pipeline applied after the raw trip files are prepared. It starts with Divvy trip validation and feature engineering, removes rule-flagged bad records, converts raw coordinates into stable vector-mapped location keys, enriches trips with nearest-station hourly weather data, and finishes by exporting the final enriched dataset for downstream analysis.

**Workflow Summary:**
1. **Divvy Data Processing**
   - Standardize datetime fields, calculate `ride_length`, round timestamps, and derive reusable time features.
   - Build reusable validation masks and dirty-data rules for chronology, missing values, and duplicate records.
   - Generate QA tables, A/B comparisons, and reconciliation summaries.

2. **Bad Data Clean Up and Drop**
   - Combine all rule flags into one dirty-row mask.
   - Split records into `dirty_data` and `clean_data`, remove invalid rows from the working dataset, and drop no-longer-needed columns.

3. **Coordinate Vector Mapping**
   - Build a geographic bounding box from trip coordinates.
   - Map start and end latitude/longitude values into stable grid-based vector keys for downstream grouping and comparison.

4. **Weather Data Processing and Merge**
   - Match each trip start grid point to the nearest weather station.
   - Merge hourly weather observations into trip records and produce merge-audit diagnostics.

5. **Export Final Enriched Data**
   - Export the final processed dataset as CSV, with optional Excel output.

**Primary outputs from this section**
- `clean_data` and `dirty_data` for traceable record separation.  
- Vector-mapped trip fields for location-based grouping.  
- `df_weather_merge` as the final weather-enriched analytical dataset.  
- QA and audit tables that document validation and merge quality.

### Divvy Data Processing - Script flow (order of operation)
1. Convert `started_at` and `ended_at` to datetime.
2. Compute `ride_length` in seconds and round to a configurable interval.
3. Round timestamps to a configurable interval for time-bucket consistency.
4. Create analysis-ready time fields (`day_of_week`, `dt_start_hour`).
5. Build a reusable masking and validation framework with modular QA components:  
   - 5.1 - Build low-level boolean masks (negative duration, end-before-start, missing location context).  
   - 5.2 - Define dirty-data business rules using those masks, expected values, and reasons.  
   - 5.3 - Map each rule to A/B example columns for transparent QA review.  
   - 5.4 - Summarize rule-level counts, percentages, union dirty count, and overlap count.  
   - 5.5 - Generate A/B showcases (dirty rows vs expected rows) for each triggered rule.  
   - 5.6 - Build location null cross-check tables for required start/end fields.  
   - 5.7 - Build a compact validation summary and reconciliation checks before cleanup.
6. Execute the validation workflow (masks -> rules -> summaries -> cross-checks).
7. Generate QA outputs (A/B samples, dirty overview, chronology/location cross-checks, validation summary).
8. Build the final union mask and split rows into `dirty_data` and `clean_data` for downstream cleanup.

**Verification outputs**
- A/B examples (dirty vs. expected) for each triggered rule.
- Chronology cross-check and location completeness breakdown.
- Validation summary totals and union-count consistency checks.
- Dirty/clean reconciliation in `cleaning_audit`.

In [15]:
# 1) Parse and normalize datetime fields used throughout validation and feature engineering.
df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce')

# 2) Compute trip duration in seconds, then round to a configurable precision.
# Lower values keep finer precision; higher values simplify durations.
ride_length_round_to_seconds = 10
ride_length_seconds = (df['ended_at'] - df['started_at']).dt.total_seconds()
df['ride_length'] = ((ride_length_seconds / ride_length_round_to_seconds).round() * ride_length_round_to_seconds).astype('Int64')

# 3) Round timestamps to a configurable interval for consistent temporal grouping.
# Example values: 10s, 15s, 30s, 60s, 300s.
round_time_to_seconds = 300
round_freq = f"{round_time_to_seconds}s"
df['started_at'] = df['started_at'].dt.round(round_freq)
df['ended_at'] = df['ended_at'].dt.round(round_freq)

# 4) Derive reusable time dimensions for downstream analysis.
# 4.1) Hour bucket from started_at for time-of-day analysis.
df['dt_start_hour'] = df['started_at'].dt.floor('h')

# 4.2) Day-of-week index where Monday=0 and Sunday=6.
df['day_of_week'] = df['started_at'].dt.dayofweek

# 5.1) Build low-level reusable boolean masks for dirty-data rules.
def build_rule_masks(df):
    negative_ride_length_mask = df['ride_length'] < 0
    end_before_start_mask = df['ended_at'] < df['started_at']
    duration_validation_mask = negative_ride_length_mask | end_before_start_mask

    location_required_cols = [
        'start_station_name', 'start_station_id', 'start_lat', 'start_lng',
        'end_station_name', 'end_station_id', 'end_lat', 'end_lng',
    ]
    location_fields_missing_mask = df[location_required_cols].isna().any(axis=1)

    return {
        'negative_ride_length_mask': negative_ride_length_mask,
        'end_before_start_mask': end_before_start_mask,
        'duration_validation_mask': duration_validation_mask,
        'location_required_cols': location_required_cols,
        'location_fields_missing_mask': location_fields_missing_mask,
    }


# 5.2) Define primary dirty-data rules with expected values and reasons.
def build_dirty_rules(df, masks):
    return {
        'Invalid trip duration chronology': {
            'mask': masks['duration_validation_mask'],
            'expected': 'ride_length > 0 seconds and ended_at >= started_at',
            'why_dirty': 'Negative duration or end before start.'
        },
        'Zero ride length': {
            'mask': df['ride_length'] == 0,
            'expected': 'ride_length > 0 seconds',
            'why_dirty': 'Zero-duration trip.'
        },
        'Missing rider type': {
            'mask': df['member_casual'].isna(),
            'expected': "member_casual in {'member','casual'}",
            'why_dirty': 'Rider segment missing.'
        },
        'Missing trip location context (start OR end)': {
            'mask': masks['location_fields_missing_mask'],
            'expected': 'All start/end location fields are populated (no nulls)',
            'why_dirty': 'At least one required location field is missing.',
        },
        'Duplicate ride_id': {
            'mask': df.duplicated(subset='ride_id', keep=False),
            'expected': 'ride_id unique per trip',
            'why_dirty': 'Duplicate trip IDs.'
        },
    }


# 5.3) Map each dirty-data rule to columns shown in A/B QA examples.
def build_ab_cols_map():
    return {
        'Invalid trip duration chronology': ['ride_id', 'started_at', 'ended_at', 'ride_length'],
        'Zero ride length': ['ride_id', 'started_at', 'ended_at', 'ride_length'],
        'Missing rider type': ['ride_id', 'member_casual'],
        'Missing trip location context (start OR end)': [
            'ride_id',
            'start_station_name', 'start_station_id', 'start_lat', 'start_lng',
            'end_station_name', 'end_station_id', 'end_lat', 'end_lng',
        ],
        'Duplicate ride_id': ['ride_id', 'started_at', 'ended_at', 'rideable_type', 'member_casual'],
    }


# 5.4) Summarize per-rule counts, plus union and overlap counts.
def summarize_dirty_rules(df, dirty_rules):
    total_rows = len(df)
    case_rows = []

    for case_name, rule in dirty_rules.items():
        count = len(df[rule['mask']])
        case_rows.append({
            'dirty_case': case_name,
            'count': count,
            'pct_of_rows': round((count / total_rows) * 100, 4),
            'expected_value': rule['expected'],
            'why_considered_dirty': rule['why_dirty'],
        })

    case_flags = pd.DataFrame({name: rule['mask'].to_numpy() for name, rule in dirty_rules.items()}, index=df.index)

    any_dirty_count = len(df[case_flags.any(axis=1)])
    multi_rule_count = len(df[case_flags.sum(axis=1) > 1])

    case_rows.append({
        'dirty_case': 'ANY dirty row (union of all rules)',
        'count': any_dirty_count,
        'pct_of_rows': round((any_dirty_count / total_rows) * 100, 4),
        'expected_value': 'At least one rule violated',
        'why_considered_dirty': 'Union count.'
    })

    case_rows.append({
        'dirty_case': 'Rows flagged by >=2 rules (overlap)',
        'count': multi_rule_count,
        'pct_of_rows': round((multi_rule_count / total_rows) * 100, 4),
        'expected_value': 'Prefer one issue per row',
        'why_considered_dirty': 'Overlap count.'
    })

    case_summary = pd.DataFrame(case_rows).sort_values('count', ascending=False)
    return case_summary, case_flags, any_dirty_count, multi_rule_count


# 5.5) Display rule-by-rule A/B examples (dirty rows vs expected rows).
def display_ab_showcase(df, dirty_rules, ab_cols_map):
    hidden_display_cols = {'ride_id', 'start_station_id', 'end_station_id'}

    for case_name, rule in dirty_rules.items():
        mask = rule['mask']
        if not mask.any():
            continue

        cols = ab_cols_map.get(case_name, ['ride_id'])
        display_cols = [col for col in cols if col not in hidden_display_cols] or cols

        print(f"\n=== {case_name} ===")
        print(f"Total flagged rows: {len(df[mask]):,}")
        print(f"Total non-flagged rows: {len(df[~mask]):,}")
        print(f"Expected: {rule['expected']}")
        print(f"Why dirty: {rule['why_dirty']}")

        a_dirty = df.loc[mask, display_cols].head(3).copy()
        a_dirty.insert(0, 'A/B', 'A (dirty)')

        b_clean = df.loc[~mask, display_cols].head(3).copy()
        b_clean.insert(0, 'A/B', 'B (clean)')

        display(pd.concat([a_dirty, b_clean], ignore_index=True))


# 5.6) Build null-count QA table for required start/end location fields.
def build_location_cross_check(df, masks):
    location_required_cols = masks['location_required_cols']
    total_rows = len(df)

    location_cross_check = pd.DataFrame({
        'field': location_required_cols,
        'missing_rows': [int(df[col].isna().sum()) for col in location_required_cols],
    })
    location_cross_check['missing_pct'] = (location_cross_check['missing_rows'] / total_rows * 100).round(4)

    return location_cross_check.sort_values('missing_rows', ascending=False)


# 5.7) Build compact QA summary for totals, completeness, and union consistency.
def build_validation_summary(df, masks, case_flags, any_dirty_count):
    total_rows = len(df)
    missing_location_rows = int(masks['location_fields_missing_mask'].sum())
    complete_location_rows = int((~masks['location_fields_missing_mask']).sum())
    union_recalc = int(case_flags.any(axis=1).sum())

    return pd.DataFrame([
        {'check': 'Total rows', 'value': total_rows},
        {'check': 'Dirty rows (union)', 'value': any_dirty_count},
        {'check': 'Rows with complete location fields', 'value': complete_location_rows},
        {'check': 'Rows with missing location fields', 'value': missing_location_rows},
        {'check': 'Union count consistent', 'value': union_recalc == any_dirty_count},
    ])


# 6) Execute the validation workflow in order: masks -> rules -> summaries -> cross-checks.
masks = build_rule_masks(df)
dirty_rules = build_dirty_rules(df, masks)
ab_cols_map = build_ab_cols_map()

case_summary, case_flags, any_dirty_count, multi_rule_count = summarize_dirty_rules(df, dirty_rules)
rule_count_check = pd.DataFrame([
    {'rule': name, 'count': len(df[rule['mask']])}
    for name, rule in dirty_rules.items()
])

chronology_cross_check = pd.DataFrame([
    {'metric': 'Negative ride length rows', 'value': int(masks['negative_ride_length_mask'].sum())},
    {'metric': 'End time before start time rows', 'value': int(masks['end_before_start_mask'].sum())},
    {'metric': 'Combined chronology-invalid rows', 'value': int(masks['duration_validation_mask'].sum())},
])

location_cross_check = build_location_cross_check(df, masks)
validation_summary = build_validation_summary(df, masks, case_flags, any_dirty_count)

# 7) Display QA outputs used to validate rule behavior and aggregate results.
display_ab_showcase(df, dirty_rules, ab_cols_map)

print('\n=== Dirty Data Overview ===')
display(case_summary)

print('\n=== Validation Summary ===')
display(validation_summary)


# 8) Materialize dirty/clean row partitions and build a one-table reconciliation audit.
combined_dirty_mask = case_flags.any(axis=1)

dirty_data = df.loc[combined_dirty_mask].copy()
clean_data = df.loc[~combined_dirty_mask].copy()

original_shape = df.shape
dirty_count = len(dirty_data)
clean_count = len(clean_data)
union_recalc = int(combined_dirty_mask.sum())
union_check_ok = union_recalc == any_dirty_count

cleaning_audit = pd.DataFrame([
    {"metric": "original_rows", "value": original_shape[0]},
    {"metric": "dirty_rows", "value": dirty_count},
    {"metric": "clean_rows", "value": clean_count},
    {"metric": "union_recalc", "value": union_recalc},
    {"metric": "union_matches_any_dirty_count", "value": union_check_ok},
])

# print("\n=== Cleaning Audit ===")
# display(cleaning_audit)

# Output variables from this cell:
# - masks (dict): reusable boolean masks for low-level validation checks.
# - dirty_rules (dict): rule definitions with mask + expected value + reason.
# - ab_cols_map (dict): columns used for A/B dirty-vs-expected examples per rule.
# - case_summary (DataFrame): per-rule counts, percentages, union, and overlap rows.
# - case_flags (DataFrame[bool]): rule-flag matrix (one boolean column per rule).
# - any_dirty_count (int): total rows flagged by at least one rule (union count).
# - multi_rule_count (int): rows flagged by two or more rules (overlap count).
# - validation_summary (DataFrame): compact totals and completeness checks.
# - rule_count_check (DataFrame): quick per-rule counts for reconciliation.
# - chronology_cross_check (DataFrame): chronology QA checks.
# - location_cross_check (DataFrame): null profile across required location fields.
# - dt_start_hour (Series[datetime]): started_at truncated to the hour for time-of-day analysis.

# Additional output variables created below:
# - combined_dirty_mask (Series[bool]): final union dirty mask from case_flags.
# - dirty_data (DataFrame): all rows flagged dirty by any rule.
# - clean_data (DataFrame): all rows passing all rules (includes rounded ride_length in seconds).
# - original_shape (tuple): shape before removing dirty rows.
# - dirty_count (int): number of dirty rows in dirty_data.
# - clean_count (int): number of clean rows in clean_data.
# - union_recalc (int): recomputed union count from combined_dirty_mask.
# - union_check_ok (bool): whether recomputed union matches any_dirty_count.
# - cleaning_audit (DataFrame): one-table reconciliation summary.



=== Zero ride length ===
Total flagged rows: 2,041
Total non-flagged rows: 667,455
Expected: ride_length > 0 seconds
Why dirty: Zero-duration trip.


,A/B,started_at,ended_at,ride_length
0,A (dirty),2025-03-01 08:45:00,2025-03-01 08:45:00,0
1,A (dirty),2025-03-01 09:25:00,2025-03-01 09:25:00,0
2,A (dirty),2025-03-01 10:15:00,2025-03-01 10:15:00,0
3,B (clean),2025-02-28 08:05:00,2025-03-01 09:05:00,89990
4,B (clean),2025-02-28 12:30:00,2025-03-01 13:30:00,89980
5,B (clean),2025-02-28 13:55:00,2025-03-01 14:55:00,89990



=== Missing trip location context (start OR end) ===
Total flagged rows: 200,800
Total non-flagged rows: 468,696
Expected: All start/end location fields are populated (no nulls)
Why dirty: At least one required location field is missing.


,A/B,start_station_name,start_lat,start_lng,end_station_name,end_lat,end_lng
0,A (dirty),Orleans St & Hubbard St,41.890028,-87.636618,NaN,NaN,NaN
1,A (dirty),Sheridan Rd & Montrose Ave,41.961670,-87.654640,NaN,NaN,NaN
2,A (dirty),DuSable Lake Shore Dr & Monroe St,41.880958,-87.616743,NaN,NaN,NaN
3,B (clean),Sheffield Ave & Webster Ave,41.921540,-87.653818,Larrabee St & Menomonee St,41.914680,-87.643320
4,B (clean),Wilton Ave & Belmont Ave,41.940232,-87.652944,Honore St & Division St,41.903119,-87.673935
5,B (clean),Michigan Ave & Madison St,41.882134,-87.625125,Wabash Ave & Adams St,41.879472,-87.625689



=== Dirty Data Overview ===


,dirty_case,count,pct_of_rows,expected_value,why_considered_dirty
5,ANY dirty row (union of all rules),201288,30.0656,At least one rule violated,Union count.
3,Missing trip location context (start OR end),200800,29.9927,All start/end location fields are populated (n...,At least one required location field is missing.
1,Zero ride length,2041,0.3049,ride_length > 0 seconds,Zero-duration trip.
6,Rows flagged by >=2 rules (overlap),1553,0.2320,Prefer one issue per row,Overlap count.
0,Invalid trip duration chronology,0,0.0000,ride_length > 0 seconds and ended_at >= starte...,Negative duration or end before start.
2,Missing rider type,0,0.0000,"member_casual in {'member','casual'}",Rider segment missing.
4,Duplicate ride_id,0,0.0000,ride_id unique per trip,Duplicate trip IDs.



=== Validation Summary ===


,check,value
0,Total rows,669496
1,Dirty rows (union),201288
2,Rows with complete location fields,468696
3,Rows with missing location fields,200800
4,Union count consistent,True


### Bad Data Clean Up and Drop
This cell operationalizes the validation framework from **Process (Section 3)** by applying the previously defined `dirty_rules` masks to the current `df`, then preparing a clean baseline for coordinate mapping in the next section.

**What this cell does**
1. Builds `combined_dirty_mask` as the union of all rule masks (index-aligned to `df`).
2. Stores flagged records in `dirty_data` for QA traceability and sample inspection.
3. Removes flagged rows from `df` and synchronizes `clean_data` with the cleaned result.
4. Prints reconciliation stats (`original_shape`, cleaned shape, rows removed).
5. Displays a compact sample of removed rows (`ride_id`, timestamps, `ride_length`, `member_casual`).
6. Drops `ended_at` from both `df` and `clean_data` because `ride_length` is already derived and used downstream.

**Outputs used downstream**
- `df`: cleaned working dataset carried into **Coordinate Mapping**.
- `clean_data`: clean snapshot aligned with `df`.
- `dirty_data`: excluded records retained for audit/QA.


In [16]:
# Bad Data Clean Up and Drop
# 1) Build a row-level union mask from all dirty rules (index-aligned to current df).
combined_dirty_mask = np.logical_or.reduce([
    rule["mask"].reindex(df.index, fill_value=False).to_numpy()
    for rule in dirty_rules.values()
])

# 2) Save flagged rows to dirty_data for audit/sample review.
dirty_data = df.loc[combined_dirty_mask].copy()

# 3) Remove flagged rows and keep only clean records in df.
original_shape = df.shape
df = df.loc[~combined_dirty_mask].copy()

# Keep clean_data synchronized with the cleaned df state.
clean_data = df.copy()

# 4) Print original shape, cleaned shape, and rows removed.
print("Original shape:", original_shape)
print("Cleaned shape: ", df.shape)
print("Rows removed:  ", int(combined_dirty_mask.sum()))

# 5) Display a sample of removed rows.
cols_to_show = ['ride_id', 'started_at', 'ended_at', 'ride_length', 'member_casual']
display(dirty_data[cols_to_show].head(10))

# 6) Drop ended_at — ride_length is already derived and ended_at is no longer required downstream.
if 'ended_at' in df.columns:
    df = df.drop(columns=['ended_at'])
if 'ended_at' in clean_data.columns:
    clean_data = clean_data.drop(columns=['ended_at'])

print("\nColumns after drop:", df.columns.tolist())


Original shape: (669496, 16)
Cleaned shape:  (468208, 16)
Rows removed:   201288


,ride_id,started_at,ended_at,ride_length,member_casual
0,5FBF1BFE4C72756F,2025-02-28 08:05:00,2025-03-01 09:05:00,89990,member
1,FDA49BB617F644B4,2025-02-28 12:30:00,2025-03-01 13:30:00,89980,casual
2,8B20E2B0D8603957,2025-02-28 13:55:00,2025-03-01 14:55:00,89990,casual
3,B7FC6F4C1F021136,2025-02-28 14:00:00,2025-03-01 15:00:00,89990,member
4,F837FF70199A73B3,2025-02-28 14:05:00,2025-03-01 15:05:00,89980,casual
5,AABE0F307027EE82,2025-02-28 14:10:00,2025-03-01 15:10:00,89980,member
6,3367A3B7A5C4A6F5,2025-02-28 15:55:00,2025-03-01 16:55:00,89990,member
7,CFA71F7010931BD2,2025-02-28 16:35:00,2025-03-01 17:35:00,90000,member
8,12B14EB8299BA751,2025-02-28 17:40:00,2025-03-01 18:40:00,90000,casual
12,B8BECDE806F69929,2025-02-28 23:45:00,2025-03-01 00:05:00,1090,member



Columns after drop: ['ride_id', 'rideable_type', 'started_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'ride_length', 'dt_start_hour', 'day_of_week']


### Coordinate Vector Mapping
Convert latitude/longitude values into stable grid-based location keys for easier grouping and comparison.

**Python script process (order of actions)**
1. Run `create_vector_map(df)` to compute the latitude/longitude bounding box from `start_*` and `end_*` coordinates, then apply a small buffer.
2. Set `grid_resolution` (in degrees) to control map granularity.
3. Run `create_grid_lock(vector_map, grid_resolution)` to build grid metadata: bounds, step counts, resolution scalar, and estimated cell edge/area sizes.
4. Run `apply_grid_mapping(df, grid_lock)` to assign each trip point to a grid-cell center using floor-based binning from the grid origin.
5. Create separate vector-mapped columns (`start-lat_vmap`, `start-lng_vmap`, `end-lat_vmap`, `end-lng_vmap`) from mapped center coordinates and remove intermediate center columns.
6. Replace `df` with the mapped output for downstream grouping and analysis.

**Result used downstream**
- Stable grid-based start/end location keys reduce coordinate noise.
- Resolution can be tuned: lower values = finer detail, higher values = coarser detail.

In [17]:
# Coordinate mapping

# 1) Create vector map (bounding box) from cleaned bike data
def create_vector_map(df):
    """Step 1: Compute bounding box from start/end lat/lng values."""
    lat_min = min(df['start_lat'].min(), df['end_lat'].min())
    lat_max = max(df['start_lat'].max(), df['end_lat'].max())
    lng_min = min(df['start_lng'].min(), df['end_lng'].min())
    lng_max = max(df['start_lng'].max(), df['end_lng'].max())

    buffer = 0.02
    vector_map = {
        'lat_north': round(lat_max + buffer, 4),
        'lat_south': round(lat_min - buffer, 4),
        'lng_west': round(lng_min - buffer, 4),
        'lng_east': round(lng_max + buffer, 4),
    }
    print('Vector map (bounding box):', vector_map)
    return vector_map


# 2) Create coordinate grid lock from vector map
def create_grid_lock(vector_map, grid_resolution=0.01):
    """Step 2: Build grid lock using resolution in degrees.

    Tuning guide:
    - Lower grid_resolution -> finer grid (more detail, more cells).
    - Higher grid_resolution -> coarser grid (less detail, fewer cells).
    """
    lat_steps = int((vector_map['lat_north'] - vector_map['lat_south']) / grid_resolution) + 1
    lng_steps = int((vector_map['lng_east'] - vector_map['lng_west']) / grid_resolution) + 1

    # Scalar view of the chosen grid resolution.
    # Example: 0.01 -> scalar 100, 0.005 -> scalar 200.
    grid_scalar = 1.0 / grid_resolution

    # Approximate cell edge length assuming 1 degree latitude ~= 111 km ~= 68.973 miles.
    approx_edge_km = 111.0 * grid_resolution
    approx_edge_m = approx_edge_km * 1000.0
    approx_edge_miles = 68.973 * grid_resolution
    approx_edge_feet = approx_edge_miles * 5280.0

    # Approximate cell area as a square footprint from the edge-length estimate.
    approx_area_sq_km = approx_edge_km ** 2
    approx_area_sq_m = approx_edge_m ** 2
    approx_area_sq_miles = approx_edge_miles ** 2
    approx_area_sq_feet = approx_edge_feet ** 2

    grid_lock = {
        'grid_resolution': grid_resolution,
        'grid_scalar': grid_scalar,
        'approx_edge_km': approx_edge_km,
        'approx_edge_m': approx_edge_m,
        'approx_edge_miles': approx_edge_miles,
        'approx_edge_feet': approx_edge_feet,
        'approx_area_sq_km': approx_area_sq_km,
        'approx_area_sq_m': approx_area_sq_m,
        'approx_area_sq_miles': approx_area_sq_miles,
        'approx_area_sq_feet': approx_area_sq_feet,
        'lat_north': vector_map['lat_north'],
        'lat_south': vector_map['lat_south'],
        'lng_west': vector_map['lng_west'],
        'lng_east': vector_map['lng_east'],
        'lat_steps': lat_steps,
        'lng_steps': lng_steps,
    }

    print(f"Grid lock created: {lat_steps} lat steps x {lng_steps} lng steps at {grid_resolution} deg resolution")
    print(f"Resolution scalar (1 / grid_resolution): {grid_scalar:.2f}")
    print(
        "Cell edge length estimate (1D side length): "
        f"{approx_edge_km:.6f} km | {approx_edge_m:.2f} m | "
        f"{approx_edge_miles:.6f} miles | {approx_edge_feet:.2f} ft"
    )
    print(
        "Cell area estimate (2D surface footprint): "
        f"{approx_area_sq_km:.8f} sq km | {approx_area_sq_m:.2f} sq m | "
        f"{approx_area_sq_miles:.8f} sq miles | {approx_area_sq_feet:.2f} sq ft"
    )
    return grid_lock


# 3) Map bike data points to grid-cell centers and create key columns
def apply_grid_mapping(df, grid_lock):
    """Step 3: Map coordinates to cell centers and create separate vmap columns per axis."""
    mapped = df.copy()
    res = grid_lock['grid_resolution']

    lat_origin = grid_lock['lat_south']
    lng_origin = grid_lock['lng_west']

    # Bin index from grid origin, then shift by +0.5 to center of each cell.
    start_lat_idx = np.floor((mapped['start_lat'] - lat_origin) / res)
    start_lng_idx = np.floor((mapped['start_lng'] - lng_origin) / res)
    end_lat_idx = np.floor((mapped['end_lat'] - lat_origin) / res)
    end_lng_idx = np.floor((mapped['end_lng'] - lng_origin) / res)

    mapped['start_lat_center'] = lat_origin + (start_lat_idx + 0.5) * res
    mapped['start_lng_center'] = lng_origin + (start_lng_idx + 0.5) * res
    mapped['end_lat_center'] = lat_origin + (end_lat_idx + 0.5) * res
    mapped['end_lng_center'] = lng_origin + (end_lng_idx + 0.5) * res

    # Separate vmap columns keep vector-mapped coordinates distinct from raw lat/lng fields.
    mapped['start-lat_vmap'] = mapped['start_lat_center'].round(6)
    mapped['start-lng_vmap'] = mapped['start_lng_center'].round(6)
    mapped['end-lat_vmap'] = mapped['end_lat_center'].round(6)
    mapped['end-lng_vmap'] = mapped['end_lng_center'].round(6)

    mapped = mapped.drop(columns=['start_lat_center', 'start_lng_center', 'end_lat_center', 'end_lng_center'])
    print(f"Mapping complete: {mapped['start-lat_vmap'].nunique():,} unique start grid-cell centers")
    return mapped


# Apply coordinate mapping to cleaned df and keep df as the main flow variable
vector_map = create_vector_map(df)

# Change this value to control map granularity.
# Lower = finer detail, Higher = coarser detail.
grid_resolution = 0.00005
grid_lock = create_grid_lock(vector_map, grid_resolution=grid_resolution)

df = apply_grid_mapping(df, grid_lock)
df[['start-lat_vmap', 'start-lng_vmap', 'end-lat_vmap', 'end-lng_vmap']].head()

Vector map (bounding box): {'lat_north': 42.0849, 'lat_south': 41.6285, 'lng_west': -87.864, 'lng_east': -87.5082}
Grid lock created: 9128 lat steps x 7117 lng steps at 5e-05 deg resolution
Resolution scalar (1 / grid_resolution): 20000.00
Cell edge length estimate (1D side length): 0.005550 km | 5.55 m | 0.003449 miles | 18.21 ft
Cell area estimate (2D surface footprint): 0.00003080 sq km | 30.80 sq m | 0.00001189 sq miles | 331.56 sq ft
Mapping complete: 2,201 unique start grid-cell centers


,start-lat_vmap,start-lng_vmap,end-lat_vmap,end-lng_vmap
9,41.921525,-87.653825,41.914675,-87.643325
10,41.940225,-87.652925,41.903125,-87.673925
11,41.882125,-87.625125,41.879475,-87.625675
15,41.872225,-87.661375,41.883175,-87.648725
16,41.883625,-87.648625,41.893925,-87.641725


In [18]:

# Coordinate-mapping checkpoint: drop raw lat/lng columns, reorder columns, and confirm vmap columns.
# Outputs:
# - df: working dataset with raw coordinate columns removed and columns reordered for downstream use.
# - vmap_cols: list of the four new grid-mapped coordinate columns.

# 1) Drop raw coordinate columns now superseded by vmap columns.
cols_to_drop_coords = ['start_lat', 'start_lng', 'end_lat', 'end_lng']
df = df.drop(columns=[c for c in cols_to_drop_coords if c in df.columns])

# 2) Reorder columns; pass-through any unexpected columns at the end.
ordered_cols = [
    'started_at',
    'day_of_week',
    'start_station_name',
    'start_station_id',
    'start-lat_vmap',
    'start-lng_vmap',
    'end_station_name',
    'end_station_id',
    'end-lat_vmap',
    'end-lng_vmap',
    'ride_id',
    'member_casual',
    'rideable_type',
    'ride_length',
    'dt_start_hour',
]
remaining_cols = [c for c in df.columns if c not in ordered_cols]
df = df[ordered_cols + remaining_cols]

# 3) Validate all expected vmap columns are present.
vmap_cols = ['start-lat_vmap', 'start-lng_vmap', 'end-lat_vmap', 'end-lng_vmap']
missing_vmap = [c for c in vmap_cols if c not in df.columns]
if missing_vmap:
    raise ValueError(f"Missing expected vmap columns after coordinate mapping: {missing_vmap}")

print(f"df shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

print("\nvmap column preview:")
display(df[vmap_cols].head(10))
display(df.head())


df shape: 468,208 rows x 15 columns

Columns (15):
  started_at
  day_of_week
  start_station_name
  start_station_id
  start-lat_vmap
  start-lng_vmap
  end_station_name
  end_station_id
  end-lat_vmap
  end-lng_vmap
  ride_id
  member_casual
  rideable_type
  ride_length
  dt_start_hour

vmap column preview:


,start-lat_vmap,start-lng_vmap,end-lat_vmap,end-lng_vmap
9,41.921525,-87.653825,41.914675,-87.643325
10,41.940225,-87.652925,41.903125,-87.673925
11,41.882125,-87.625125,41.879475,-87.625675
15,41.872225,-87.661375,41.883175,-87.648725
16,41.883625,-87.648625,41.893925,-87.641725
17,41.885775,-87.651025,41.894725,-87.634375
18,41.894975,-87.632425,41.894575,-87.653425
19,41.952175,-87.698075,41.965925,-87.693625
20,41.952375,-87.677275,41.954175,-87.704425
21,42.032575,-87.679125,42.010575,-87.662425


,started_at,day_of_week,start_station_name,start_station_id,start-lat_vmap,start-lng_vmap,end_station_name,end_station_id,end-lat_vmap,end-lng_vmap,ride_id,member_casual,rideable_type,ride_length,dt_start_hour
9,2025-02-28 19:20:00,4,Sheffield Ave & Webster Ave,TA1309000033,41.921525,-87.653825,Larrabee St & Menomonee St,TA1306000007,41.914675,-87.643325,2C8C21C83E2E7B07,casual,classic_bike,60150,2025-02-28 19:00:00
10,2025-02-28 22:35:00,4,Wilton Ave & Belmont Ave,TA1307000134,41.940225,-87.652925,Honore St & Division St,TA1305000034,41.903125,-87.673925,49DADDCD5CB4CEE4,member,electric_bike,6590,2025-02-28 22:00:00
11,2025-02-28 23:25:00,4,Michigan Ave & Madison St,13036,41.882125,-87.625125,Wabash Ave & Adams St,KA1503000015,41.879475,-87.625675,D5A9EF72ED166C2E,member,classic_bike,2690,2025-02-28 23:00:00
15,2025-02-28 23:50:00,4,Loomis St & Lexington St,13332,41.872225,-87.661375,Green St & Washington Blvd,13053,41.883175,-87.648725,F8CC20B031C350CA,member,classic_bike,540,2025-02-28 23:00:00
16,2025-02-28 23:50:00,4,Green St & Randolph St*,chargingstx3,41.883625,-87.648625,Kingsbury St & Erie St,13265,41.893925,-87.641725,F455B02969E1D86E,member,classic_bike,500,2025-02-28 23:00:00


### Weather Data Processing and Merge
**Tools of choice**<br>
Python with pandas and numpy — scalable, reproducible, and suitable for large trip and hourly weather datasets.

**Script flow (order of operation)**
1. Parse `weather_df['time']` from Open-Meteo ISO format (`YYYY-MM-DDTHH:MM`) into hourly Python datetime values and convert `df['dt_start_hour']` to datetime for joining.
2. Drop any previously merged weather-enrichment columns from `df` so the cell can be rerun safely without duplication.
3. Build weather enrichment fields using imperial outputs and feet-based height labels: `temperature_7ft_f`, `relative_humidity_7ft_pct`, `precipitation_in`, `rain_in`, `snowfall_in`, `wind_speed_33ft_mph`, `wind_direction_33ft_deg`, `cloud_cover_pct`.
4. Extract all unique combinations of `start-lat_vmap` and `start-lng_vmap` from `df` as trip-origin grid points.
5. Build a Cartesian join between each unique trip start point and every weather location in `weather_locations_df`, then compute haversine great-circle distances in kilometers (and miles) for each pair.
6. Select the single nearest weather location for each unique trip start point (`start-lat_vmap`/`start-lng_vmap`) and store the result in `start_weather_map`.
7. Merge `start_weather_map` into `df` to attach the nearest weather location ID and proximity metrics to every bike trip row.
8. Merge hourly weather observations from `weather_df` into `df` on `weather_location_id` + `dt_start_hour`, keeping imperial weather outputs plus dimensionless context fields (% and compass degrees).
9. Identify unmatched rows after the merge and classify each into `unmatched_reason` using ordered priority: missing trip hour -> missing weather location mapping -> before coverage window -> after coverage window -> no hourly row for mapped location/hour.
10. Build geo-mapping QA tables (`weather_location_assignment_summary`, `weather_geo_mapping_audit`) to verify which `location_id` each `start-lat_vmap`/`start-lng_vmap` grid point is assigned to and how far the assigned weather station is from the trip origin.
11. Compile and display a compact `weather_merge_audit` summary with unmatched diagnostics and mapping QA outputs.
12. Create `df_weather_merge` as the final weather-enriched bike dataset and remove any legacy duplicate weather columns if present.

**Important handling note**
- Unmatched bike rows are retained in `df` (left-join behavior) so row count is preserved for downstream analysis.
- For those retained rows, weather enrichment fields remain null while `unmatched_reason` explains why weather was not attached.

**Outputs created in this step**
- `weather_df` — weather observations with parsed datetime and added imperial/feet-labeled enrichment fields.
- `start_weather_map` — lookup from each unique bike start grid point to its nearest weather location, including proximity metrics.
- `df` — enriched bike dataset with nearest weather location plus hourly weather outputs.
- `weather_merge_audit` — compact merge validation summary (row counts, coverage window, matched vs. missing).
- `unmatched_weather_rows_df` — rows missing weather data after the merge, with categorized reason fields.
- `unmatched_weather_summary` — counts of unmatched rows grouped by reason.
- `weather_location_assignment_summary` — grouped counts of bike rows mapped to each weather `location_id`.
- `weather_geo_mapping_audit` — QA table showing proximity between each `start-lat_vmap`/`start-lng_vmap` grid point and its assigned weather coordinates.
- `df_weather_merge` — final weather-enriched dataset carried into downstream grouping/export.

In [19]:
# ## Weather Data Processing and Merge
# 1) Parse weather timestamps and trip start-hour values into join-ready datetime fields.
weather_df = weather_df.copy()
weather_locations_df = weather_locations_df.copy()

weather_df['time'] = pd.to_datetime(weather_df['time'], errors='coerce')
df['dt_start_hour'] = pd.to_datetime(df['dt_start_hour'], errors='coerce')

# 2) Drop previously merged weather-enrichment columns so this cell can be rerun safely.
weather_enrichment_cols = [
    'weather_location_id',
    'weather_latitude',
    'weather_longitude',
    'distance_km',
    'distance_miles',
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'precipitation (mm)',
    'rain (mm)',
    'snowfall (cm)',
    'wind_speed_10m (km/h)',
    'wind_direction_10m (°)',
    'cloud_cover (%)',
    'temperature_7ft_f',
    'precipitation_in',
    'rain_in',
    'snowfall_in',
    'wind_speed_33ft_mph',
    'relative_humidity_7ft_pct',
    'wind_direction_33ft_deg',
    'cloud_cover_pct',
]
existing_weather_cols = [col for col in weather_enrichment_cols if col in df.columns]
if existing_weather_cols:
    df = df.drop(columns=existing_weather_cols)

# 3) Build imperial weather fields and feet-based height labels used in downstream outputs.
# Height label conversions used in column names:
# - 2 meters ~= 6.56 feet -> labeled as 7ft
# - 10 meters ~= 32.81 feet -> labeled as 33ft
weather_df['temperature_7ft_f'] = (weather_df['temperature_2m (°C)'] * 9 / 5) + 32
weather_df['precipitation_in'] = weather_df['precipitation (mm)'] / 25.4
weather_df['rain_in'] = weather_df['rain (mm)'] / 25.4
weather_df['snowfall_in'] = weather_df['snowfall (cm)'] / 2.54
weather_df['wind_speed_33ft_mph'] = weather_df['wind_speed_10m (km/h)'] * 0.621371
weather_df['relative_humidity_7ft_pct'] = weather_df['relative_humidity_2m (%)']
weather_df['wind_direction_33ft_deg'] = weather_df['wind_direction_10m (°)']
weather_df['cloud_cover_pct'] = weather_df['cloud_cover (%)']

# 4) Extract unique trip start grid points from separate vmap columns.
start_points = (
    df[['start-lat_vmap', 'start-lng_vmap']]
    .drop_duplicates()
    .rename(columns={
        'start-lat_vmap': 'start_lat_vmap',
        'start-lng_vmap': 'start_lng_vmap',
    })
    .copy()
)

# 5) Build all start-point x weather-location candidates for nearest-station selection.
weather_locations = weather_locations_df[['location_id', 'latitude', 'longitude']].copy()
start_points['__merge_key'] = 1
weather_locations['__merge_key'] = 1
start_weather_candidates = start_points.merge(weather_locations, on='__merge_key', how='left')

# 5.1) Convert degrees to radians so haversine distance can be calculated correctly.
earth_radius_km = 6371.0
start_lat_rad = np.radians(start_weather_candidates['start_lat_vmap'])
start_lng_rad = np.radians(start_weather_candidates['start_lng_vmap'])
weather_lat_rad = np.radians(start_weather_candidates['latitude'])
weather_lng_rad = np.radians(start_weather_candidates['longitude'])

# 5.2) Compute haversine distance between each trip start point and candidate weather location.
delta_lat = weather_lat_rad - start_lat_rad
delta_lng = weather_lng_rad - start_lng_rad
haversine_a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(start_lat_rad) * np.cos(weather_lat_rad) * np.sin(delta_lng / 2) ** 2
)
haversine_c = 2 * np.arctan2(np.sqrt(haversine_a), np.sqrt(1 - haversine_a))
start_weather_candidates['distance_km'] = earth_radius_km * haversine_c
start_weather_candidates['distance_miles'] = start_weather_candidates['distance_km'] * 0.621371

# 6) Keep the single nearest weather location for each unique trip start grid point.
nearest_idx = (
    start_weather_candidates
    .groupby(['start_lat_vmap', 'start_lng_vmap'])['distance_km']
    .idxmin()
)
start_weather_map = (
    start_weather_candidates.loc[nearest_idx, [
        'start_lat_vmap',
        'start_lng_vmap',
        'location_id',
        'latitude',
        'longitude',
        'distance_km',
        'distance_miles',
    ]]
    .rename(columns={
        'start_lat_vmap': 'start-lat_vmap',
        'start_lng_vmap': 'start-lng_vmap',
        'location_id': 'weather_location_id',
        'latitude': 'weather_latitude',
        'longitude': 'weather_longitude',
    })
    .reset_index(drop=True)
)

# 7) Attach nearest weather location metadata and proximity fields to each bike-trip row.
original_row_count = len(df)
df = df.merge(start_weather_map, on=['start-lat_vmap', 'start-lng_vmap'], how='left')

# 8) Merge hourly weather by weather_location_id + dt_start_hour using imperial/dimensionless outputs.
# Left join preserves all bike rows even when hourly weather is unavailable.
weather_merge_cols = [
    'location_id',
    'time',
    'temperature_7ft_f',
    'relative_humidity_7ft_pct',
    'precipitation_in',
    'rain_in',
    'snowfall_in',
    'wind_speed_33ft_mph',
    'wind_direction_33ft_deg',
    'cloud_cover_pct',
]

df = df.merge(
    weather_df[weather_merge_cols],
    left_on=['weather_location_id', 'dt_start_hour'],
    right_on=['location_id', 'time'],
    how='left',
).drop(columns=['location_id', 'time'])

# 9) Identify unmatched rows and classify why weather data did not join.
weather_coverage_start = weather_df['time'].min()
weather_coverage_end = weather_df['time'].max()

# Unmatched rows are intentionally retained in df; this QA frame isolates them for diagnostics only.
unmatched_weather_rows_df = df.loc[df['temperature_7ft_f'].isna()].copy()
# Classification is ordered by priority (first true condition wins).
unmatched_weather_rows_df['unmatched_reason'] = np.select(
    [
        unmatched_weather_rows_df['dt_start_hour'].isna(),
        unmatched_weather_rows_df['weather_location_id'].isna(),
        unmatched_weather_rows_df['dt_start_hour'] < weather_coverage_start,
        unmatched_weather_rows_df['dt_start_hour'] > weather_coverage_end,
    ],
    [
        'Missing dt_start_hour',
        'Missing weather location mapping',
        'Before weather coverage window',
        'After weather coverage window',
    ],
    default='No hourly weather row for mapped location and hour',
)

unmatched_weather_summary = (
    unmatched_weather_rows_df['unmatched_reason']
    .value_counts(dropna=False)
    .rename_axis('unmatched_reason')
    .reset_index(name='row_count')
)

# Add condition-level definitions and recommended next actions for each unmatched reason.
unmatched_reason_guide = pd.DataFrame([
    {
        'unmatched_reason': 'Missing dt_start_hour',
        'condition_detail': 'Trip hour key is null after datetime parsing/rounding.',
        'suggested_fix': 'Review started_at parsing and dt_start_hour derivation; repair or drop rows with invalid timestamps.',
    },
    {
        'unmatched_reason': 'Missing weather location mapping',
        'condition_detail': 'No nearest weather location_id was attached to the start grid point.',
        'suggested_fix': 'Validate start-lat_vmap/start-lng_vmap values and weather location table coverage for the study area.',
    },
    {
        'unmatched_reason': 'Before weather coverage window',
        'condition_detail': 'Trip hour is earlier than the minimum weather_df time.',
        'suggested_fix': 'Expand weather_start_date backward or limit bike rows to covered dates.',
    },
    {
        'unmatched_reason': 'After weather coverage window',
        'condition_detail': 'Trip hour is later than the maximum weather_df time.',
        'suggested_fix': 'Expand weather_end_date forward or limit bike rows to covered dates.',
    },
    {
        'unmatched_reason': 'No hourly weather row for mapped location and hour',
        'condition_detail': 'Station is mapped but no hourly observation exists for that exact hour.',
        'suggested_fix': 'Inspect weather source gaps, timezone alignment, and consider fallback joins (nearest available hour).',
    },
])

unmatched_weather_summary_detailed = (
    unmatched_weather_summary
    .merge(unmatched_reason_guide, on='unmatched_reason', how='left')
)

# 10) Build geo-mapping QA tables for station assignment counts and point-to-station proximity.
weather_location_assignment_summary = (
    df.groupby('weather_location_id', dropna=False)
    .agg(
        mapped_rows=('weather_location_id', 'size'),
        avg_distance_km=('distance_km', 'mean'),
        max_distance_km=('distance_km', 'max'),
    )
    .reset_index()
)

unique_start_points_by_location = (
    df.groupby('weather_location_id', dropna=False)[['start-lat_vmap', 'start-lng_vmap']]
    .apply(lambda g: g.drop_duplicates().shape[0])
    .rename('unique_start_points')
    .reset_index()
)

weather_location_assignment_summary = (
    weather_location_assignment_summary
    .merge(unique_start_points_by_location, on='weather_location_id', how='left')
    .sort_values('mapped_rows', ascending=False)
)

weather_geo_mapping_audit = (
    start_weather_map[[
        'start-lat_vmap',
        'start-lng_vmap',
        'weather_location_id',
        'weather_latitude',
        'weather_longitude',
        'distance_km',
        'distance_miles',
    ]]
    .sort_values('distance_km', ascending=False)
    .reset_index(drop=True)
)

# 11) Compile and display merge audit + unmatched diagnostics + mapping QA tables.
matched_weather_rows = int(df['temperature_7ft_f'].notna().sum())
missing_weather_rows = int(df['temperature_7ft_f'].isna().sum())

weather_merge_audit = pd.DataFrame([
    {'check': 'Original bike rows', 'value': original_row_count},
    {'check': 'Rows after weather merge', 'value': len(df)},
    {'check': 'Row count preserved', 'value': len(df) == original_row_count},
    {'check': 'Rows with matched weather', 'value': matched_weather_rows},
    {'check': 'Rows missing weather', 'value': missing_weather_rows},
    {'check': 'Unique mapped weather locations', 'value': int(df['weather_location_id'].nunique())},
    {'check': 'Weather coverage start', 'value': weather_coverage_start},
    {'check': 'Weather coverage end', 'value': weather_coverage_end},
])

print("=== Weather Merge Audit ===")
print(
    f"Review of merge integrity and coverage.\n"
    f"Configured weather_start_date: {weather_start_date}\n"
    f"Observed weather coverage window: {weather_coverage_start} to {weather_coverage_end}."
)
display(weather_merge_audit)

print("=== Unmatched Weather Diagnostics ===")
if unmatched_weather_rows_df.empty:
    print("All bike rows successfully matched to hourly weather data.\n")
else:
    print(
        "⚠️ Some bike rows did not match weather data. "
        "See reason counts, definitions, and suggested fixes below."
    )
    display(unmatched_weather_summary_detailed)
    print(
        "Sample unmatched rows (first 20): includes trip time key, mapped weather location,\n"
        "distance to station, and the classified unmatched_reason."
    )
    display(unmatched_weather_rows_df[[
        'started_at',
        'dt_start_hour',
        'start_station_name',
        'start-lat_vmap',
        'start-lng_vmap',
        'weather_location_id',
        'distance_km',
        'unmatched_reason',
    ]].head(20))

print("=== Weather Location Assignment Summary ===")
print(
    "Counts and distance stats by mapped weather location_id.\n"
    "Useful for identifying overused stations or far mappings."
)
display(weather_location_assignment_summary)

print("=== Geo Mapping Audit (Top 20 Farthest Start-Point Mappings) ===")
print(
    "Point-to-station mapping QA: each row is a unique trip start grid point mapped\n"
    "to its nearest weather location, sorted by largest distance_km."
)
display(weather_geo_mapping_audit.head(20))

# 12) Create final weather-enriched output and remove any legacy duplicate weather columns if present.
df_weather_merge = df.copy()

# Drop legacy/duplicate weather columns if present.
cols_to_drop = [
    'distance_km',
    'temperature_2m_f',
    'relative_humidity_2m_pct',
    'wind_speed_10m_mph',
    'wind_direction_10m_deg',
]
df_weather_merge = df_weather_merge.drop(columns=cols_to_drop, errors='ignore')

print("=== Final Weather-Enriched Dataset ===")
print(f"Rows: {len(df_weather_merge):,} | Columns: {df_weather_merge.shape[1]:,}")

# Preview rows with all columns visible.
with pd.option_context(
    'display.max_columns', None,
    'display.width', None,
    'display.max_colwidth', None
):
    display(df_weather_merge.head(10))


=== Weather Merge Audit ===
Review of merge integrity and coverage.
Configured weather_start_date: 2025-02-28
Observed weather coverage window: 2025-02-28 00:00:00 to 2025-04-30 23:00:00.


,check,value
0,Original bike rows,468208
1,Rows after weather merge,468208
2,Row count preserved,True
3,Rows with matched weather,468208
4,Rows missing weather,0
5,Unique mapped weather locations,2
6,Weather coverage start,2025-02-28 00:00:00
7,Weather coverage end,2025-04-30 23:00:00


=== Unmatched Weather Diagnostics ===
All bike rows successfully matched to hourly weather data.

=== Weather Location Assignment Summary ===
Counts and distance stats by mapped weather location_id.
Useful for identifying overused stations or far mappings.


,weather_location_id,mapped_rows,avg_distance_km,max_distance_km,unique_start_points
0,0,239125,27.385704,37.141964,3873
1,1,229083,28.007289,37.880713,3215


=== Geo Mapping Audit (Top 20 Farthest Start-Point Mappings) ===
Point-to-station mapping QA: each row is a unique trip start grid point mapped
to its nearest weather location, sorted by largest distance_km.


,start-lat_vmap,start-lng_vmap,weather_location_id,weather_latitude,weather_longitude,distance_km,distance_miles
0,41.993025,-87.821675,1,42.07381,-87.37610,37.880713,23.537977
1,41.996975,-87.821325,1,42.07381,-87.37610,37.749699,23.456568
2,41.990625,-87.816925,1,42.07381,-87.37610,37.565201,23.341927
3,42.002325,-87.817075,1,42.07381,-87.37610,37.275028,23.161621
4,41.988575,-87.812525,1,42.07381,-87.37610,37.270969,23.159099
5,42.011475,-87.819025,1,42.07381,-87.37610,37.227019,23.131790
6,42.011475,-87.818225,1,42.07381,-87.37610,37.162113,23.091460
7,41.985075,-87.823175,0,41.65202,-87.78903,37.141964,23.078939
8,41.987325,-87.810225,1,42.07381,-87.37610,37.123384,23.067394
9,42.000325,-87.812425,1,42.07381,-87.37610,36.949032,22.959057


=== Final Weather-Enriched Dataset ===
Rows: 468,208 | Columns: 27


,started_at,day_of_week,start_station_name,start_station_id,start-lat_vmap,start-lng_vmap,end_station_name,end_station_id,end-lat_vmap,end-lng_vmap,ride_id,member_casual,rideable_type,ride_length,dt_start_hour,weather_location_id,weather_latitude,weather_longitude,distance_miles,temperature_7ft_f,relative_humidity_7ft_pct,precipitation_in,rain_in,snowfall_in,wind_speed_33ft_mph,wind_direction_33ft_deg,cloud_cover_pct
0,2025-02-28 19:20:00,4,Sheffield Ave & Webster Ave,TA1309000033,41.921525,-87.653825,Larrabee St & Menomonee St,TA1306000007,41.914675,-87.643325,2C8C21C83E2E7B07,casual,classic_bike,60150,2025-02-28 19:00:00,1,42.07381,-87.37610,17.722198,40.46,63,0.0,0.0,0.0,30.074356,303,98
1,2025-02-28 22:35:00,4,Wilton Ave & Belmont Ave,TA1307000134,41.940225,-87.652925,Honore St & Division St,TA1305000034,41.903125,-87.673925,49DADDCD5CB4CEE4,member,electric_bike,6590,2025-02-28 22:00:00,1,42.07381,-87.37610,16.946416,35.42,67,0.0,0.0,0.0,24.295606,311,97
2,2025-02-28 23:25:00,4,Michigan Ave & Madison St,13036,41.882125,-87.625125,Wabash Ave & Adams St,KA1503000015,41.879475,-87.625675,D5A9EF72ED166C2E,member,classic_bike,2690,2025-02-28 23:00:00,0,41.65202,-87.78903,18.003188,37.04,59,0.0,0.0,0.0,11.246815,307,100
3,2025-02-28 23:50:00,4,Loomis St & Lexington St,13332,41.872225,-87.661375,Green St & Washington Blvd,13053,41.883175,-87.648725,F8CC20B031C350CA,member,classic_bike,540,2025-02-28 23:00:00,0,41.65202,-87.78903,16.576208,37.04,59,0.0,0.0,0.0,11.246815,307,100
4,2025-02-28 23:50:00,4,Green St & Randolph St*,chargingstx3,41.883625,-87.648625,Kingsbury St & Erie St,13265,41.893925,-87.641725,F455B02969E1D86E,member,classic_bike,500,2025-02-28 23:00:00,0,41.65202,-87.78903,17.562117,37.04,59,0.0,0.0,0.0,11.246815,307,100
5,2025-02-28 23:55:00,4,Sangamon St & Lake St,TA1306000015,41.885775,-87.651025,Wells St & Huron St,TA1306000012,41.894725,-87.634375,200F36772DBD58D1,casual,electric_bike,1060,2025-02-28 23:00:00,0,41.65202,-87.78903,17.647327,37.04,59,0.0,0.0,0.0,11.246815,307,100
6,2025-02-28 23:55:00,4,LaSalle Dr & Huron St,KP1705001026,41.894975,-87.632425,Carpenter St & Huron St,13196,41.894575,-87.653425,8325A9FC66FE4595,member,electric_bike,380,2025-02-28 23:00:00,1,42.07381,-87.37610,18.055016,34.16,71,0.0,0.0,0.0,24.482017,318,100
7,2025-02-28 23:55:00,4,California Ave & Byron St,15628,41.952175,-87.698075,Rockwell St & Eastwood Ave,KA1504000093,41.965925,-87.693625,480748E33828AB42,member,classic_bike,520,2025-02-28 23:00:00,1,42.07381,-87.37610,18.542720,34.16,71,0.0,0.0,0.0,24.482017,318,100
8,2025-02-28 23:55:00,4,Lincoln Ave & Byron St,23002,41.952375,-87.677275,Whipple/Irving Park,475,41.954175,-87.704425,AE23BB2264DC2095,casual,electric_bike,400,2025-02-28 23:00:00,1,42.07381,-87.37610,17.590946,34.16,71,0.0,0.0,0.0,24.482017,318,100
9,2025-02-28 23:55:00,4,Chicago Ave & Washington St,E002,42.032575,-87.679125,Sheridan Rd & Greenleaf Ave,KA1504000159,42.010575,-87.662425,AEF1C3423D38AE36,member,electric_bike,610,2025-02-28 23:00:00,1,42.07381,-87.37610,15.805109,34.16,71,0.0,0.0,0.0,24.482017,318,100


### Export Final Enriched Data
This step writes the final weather-enriched dataset to disk for downstream analysis.

**Purpose of this Python script**
1. Export only `df_weather_merge`.
2. Build output names from existing variables: `start_yyyymm` and `end_yyyymm`.
3. Always write CSV as `[start_yyyymm]-[end_yyyymm]-divvy-tripdata-enriched.csv`.
4. Optionally write Excel as `[start_yyyymm]-[end_yyyymm]-divvy-tripdata-enriched.xlsx` when enabled.

**Optional Excel output**
- Set `export_xlsx = True` in the setup/import configuration cell near the top of the notebook.
- CSV remains the default and is always exported.

In [20]:
# Export Final Enriched Data (df_weather_merge only)
# 1) Validate required variables exist.
required_vars = ['df_weather_merge', 'start_yyyymm', 'end_yyyymm', 'OUTPUT_DIR']
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(f"Missing required variables for export: {missing_vars}")

# 2) Build output names from pre-existing date-range variables.
base_filename = f"{start_yyyymm}-{end_yyyymm}-divvy-tripdata-enriched"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_output_path = OUTPUT_DIR / f"{base_filename}.csv"
xlsx_output_path = OUTPUT_DIR / f"{base_filename}.xlsx"

# 3) Read optional Excel-export toggle from the setup/import configuration cell.
#    export_xlsx is defined near the top of the notebook.

# 4) Export CSV output.
df_weather_merge.to_csv(csv_output_path, index=False)
print("Export source: df_weather_merge")
print(f"Rows exported: {len(df_weather_merge):,}")
print(f"Columns exported: {len(df_weather_merge.columns)}")
print(f"CSV exported to: {csv_output_path}")

# 5) Optionally export .xlsx output.
if export_xlsx:
    try:
        df_weather_merge.to_excel(xlsx_output_path, index=False)
        print(f"XLSX exported to: {xlsx_output_path}")
    except ImportError as exc:
        raise ImportError(
            "Excel export requires an engine like openpyxl. "
            "Install it in this environment, then rerun the cell. "
            f"Original error: {exc}"
        )

Export source: df_weather_merge
Rows exported: 468,208
Columns exported: 27
CSV exported to: /home/stubb/Code/articles/case-study_bike-share-success/202503-202504-divvy-tripdata-enriched.csv
